# Phishing Attack Detection — Reproduction and Critical Evaluation

This notebook reproduces and critically evaluates the GitHub project:

**RimTouny / Phishing-Attack-Detection-using-Machine-Learning**

Course context: Final Project — Data Science in Cyber.

Main task: classify URLs as **phishing** or **benign** using URL-based features.

## 1. Setup

The original notebook installs many heavy packages and pins `scikit-learn==0.24`, which may break on newer Python versions.
For this reproduction, we start with a minimal, reproducible stack.

If you want to reproduce the author's exact LazyClassifier / CatBoost / XGBoost experiments, add those packages later.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

## 2. Data Loading

The original repository does **not** include the final CSV file.  
The original notebook uses a local path:

```python
data_path = r"E:\data\data_Features (2).csv"
```

For our reproduction, place the dataset at:

```text
data/data_features.csv
```

or change `DATA_PATH` below.

In [ ]:
DATA_PATH = Path("data/data_features.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download/recreate the feature CSV and place it under data/data_features.csv. "
        "The original GitHub notebook used a local Windows path and did not provide the CSV directly."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 3. Data Inspection

We check:
- dataset size
- feature types
- missing values
- duplicate rows
- duplicated features
- single-value / near-constant features
- whether the index and columns make sense

In [ ]:
def inspect_dataset(data: pd.DataFrame, target_col: str = "label") -> None:
    print("Shape:", data.shape)
    print("\nColumns:")
    print(list(data.columns))

    print("\nData types:")
    display(data.dtypes.value_counts())

    print("\nMissing values:")
    missing = data.isna().sum().sort_values(ascending=False)
    display(missing[missing > 0])

    print("\nDuplicate rows:", data.duplicated().sum())

    print("\nTarget distribution:")
    if target_col in data.columns:
        display(data[target_col].value_counts(dropna=False))
        display((data[target_col].value_counts(normalize=True) * 100).round(2))

    print("\nSingle-value columns:")
    nunique = data.nunique(dropna=False).sort_values()
    display(nunique[nunique <= 1])

inspect_dataset(df)

In [ ]:
# Check duplicated columns/features.
def find_duplicate_columns(data: pd.DataFrame):
    duplicate_cols = []
    cols = data.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if data[cols[i]].equals(data[cols[j]]):
                duplicate_cols.append((cols[i], cols[j]))
    return duplicate_cols

duplicate_columns = find_duplicate_columns(df)
duplicate_columns

## 4. Temporal Analysis

The assignment asks for temporal analysis.  
This dataset appears to be based on URL lexical/statistical features. If there is no date/time column, we must explicitly state that temporal analysis is not possible and that this is a limitation.

In [ ]:
# Detect possible temporal columns.
possible_time_cols = []
for col in df.columns:
    lower = col.lower()
    if any(token in lower for token in ["date", "time", "timestamp", "created", "updated"]):
        possible_time_cols.append(col)

print("Possible temporal columns:", possible_time_cols)

if not possible_time_cols:
    print("No temporal columns were found. Temporal drift cannot be evaluated directly.")
else:
    for col in possible_time_cols:
        parsed = pd.to_datetime(df[col], errors="coerce")
        print(col, "valid datetime ratio:", parsed.notna().mean())

## 5. Exploratory Data Analysis (EDA)

We analyze:
- class imbalance / prevalence
- feature distributions
- outliers
- correlations

For correlation, we use **Spearman correlation** because many URL-count features are discrete, skewed, and not normally distributed. Spearman measures monotonic relationships using ranks, so it is more robust to outliers than Pearson.

In [ ]:
target_col = "label"
assert target_col in df.columns, "The dataset must contain a 'label' column."

class_counts = df[target_col].value_counts().sort_index()
class_percent = df[target_col].value_counts(normalize=True).sort_index() * 100

display(pd.DataFrame({"count": class_counts, "percent": class_percent.round(2)}))

plt.figure(figsize=(5, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Class Distribution")
plt.xlabel("Label: 0=Benign, 1=Phishing")
plt.ylabel("Count")
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(target_col, errors="ignore").tolist()

# Summary statistics for numeric features.
display(df[numeric_cols].describe().T)

# Plot a few important URL lexical features if they exist.
important_features = [
    "url_length", "domain_length", "digits", "letters",
    "path_count", "entropy", "ratio", "unusual_character_count",
    "malicious_probability"
]
important_features = [c for c in important_features if c in df.columns]

for col in important_features[:8]:
    plt.figure(figsize=(6, 4))
    plt.hist(df.loc[df[target_col] == 0, col].dropna(), bins=50, alpha=0.6, label="Benign")
    plt.hist(df.loc[df[target_col] == 1, col].dropna(), bins=50, alpha=0.6, label="Phishing")
    plt.title(f"Distribution of {col} by class")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

In [ ]:
# Outlier analysis using IQR.
def outlier_rate_iqr(series: pd.Series) -> float:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).mean()

outlier_rates = pd.Series({col: outlier_rate_iqr(df[col]) for col in numeric_cols})
display((outlier_rates.sort_values(ascending=False) * 100).round(2).head(15))

In [ ]:
# Spearman correlation with the target.
corr = df[numeric_cols + [target_col]].corr(method="spearman")[target_col].drop(target_col)
corr_sorted = corr.reindex(corr.abs().sort_values(ascending=False).index)
display(corr_sorted.head(20))

plt.figure(figsize=(8, 6))
corr_sorted.head(15).sort_values().plot(kind="barh")
plt.title("Top Spearman Correlations with Target")
plt.xlabel("Spearman correlation")
plt.show()

## 6. Feature Engineering

The original project used URL-based features such as URL length, domain length, digit count, entropy, suspicious TLD indicator, and special-character ratios.

For our reproduction:
- remove constant features
- optionally remove near-constant features
- keep numeric lexical features
- scale features for Logistic Regression
- do not scale for Random Forest / tree-based models
- apply train/test split before any resampling to avoid data leakage

Important critical note: if `malicious_probability` is computed using external labels or a prior detector, it may cause information leakage. We therefore test models both **with** and **without** this feature.

In [ ]:
def prepare_features(data: pd.DataFrame, target_col: str = "label", drop_cols=None):
    if drop_cols is None:
        drop_cols = []

    y = data[target_col].astype(int)
    X = data.drop(columns=[target_col] + drop_cols, errors="ignore").copy()

    # Keep numeric columns only for this reproduction.
    X = X.select_dtypes(include=[np.number])

    # Remove constant columns.
    constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
    X = X.drop(columns=constant_cols)

    # Fill missing numeric values with median.
    X = X.fillna(X.median(numeric_only=True))

    return X, y, constant_cols

X, y, constant_cols = prepare_features(df)
print("Removed constant columns:", constant_cols)
print("X shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train prevalence:")
display(y_train.value_counts(normalize=True).sort_index().round(4))

## 7. Model Training

The assignment requires at least two models.  
We train three baseline models:

1. Logistic Regression — interpretable linear baseline.
2. Random Forest — non-linear tree ensemble.
3. HistGradientBoosting — efficient boosting model.

Later we can add XGBoost/LightGBM/CatBoost if needed.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.08,
        random_state=RANDOM_STATE
    )
}

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = None

    result = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score) if y_score is not None else np.nan
    }

    print("\n", "=" * 80)
    print(name)
    print("=" * 80)
    print(classification_report(y_test, y_pred, digits=4))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Benign", "Phishing"])
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

    return result, model

results = []
fitted_models = {}

for name, model in models.items():
    result, fitted = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(result)
    fitted_models[name] = fitted

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)

## 8. Reproduction Check Without `malicious_probability`

This is a critical experiment.  
If performance drops strongly after removing `malicious_probability`, then this feature may dominate the model and should be discussed carefully.

In [ ]:
if "malicious_probability" in df.columns:
    X_no_mp, y_no_mp, constant_cols_no_mp = prepare_features(
        df, target_col="label", drop_cols=["malicious_probability"]
    )

    X_train2, X_test2, y_train2, y_test2 = train_test_split(
        X_no_mp, y_no_mp,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y_no_mp
    )

    results_no_mp = []
    for name, model in models.items():
        result, _ = evaluate_model(name + " without malicious_probability", model, X_train2, X_test2, y_train2, y_test2)
        results_no_mp.append(result)

    results_no_mp_df = pd.DataFrame(results_no_mp).sort_values("f1", ascending=False)
    display(results_no_mp_df)
else:
    print("Column malicious_probability was not found.")

## 9. Error Analysis

We inspect false positives and false negatives.

Cybersecurity interpretation:
- False Positive: benign URL classified as phishing. This can annoy users or block legitimate activity.
- False Negative: phishing URL classified as benign. This is usually more dangerous because the user may visit a malicious URL.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

y_pred = best_model.predict(X_test)
errors = X_test.copy()
errors["true_label"] = y_test.values
errors["pred_label"] = y_pred

false_positives = errors[(errors["true_label"] == 0) & (errors["pred_label"] == 1)]
false_negatives = errors[(errors["true_label"] == 1) & (errors["pred_label"] == 0)]

print("Best model:", best_model_name)
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

display(false_positives.head(10))
display(false_negatives.head(10))

## 10. Save Results

These outputs can be used in the final report.

In [ ]:
Path("results").mkdir(exist_ok=True)
results_df.to_csv("results/model_metrics.csv", index=False)
print("Saved results/model_metrics.csv")

## 11. Initial Conclusions Template

Fill this after running the notebook:

- The original project addresses an important phishing URL classification problem.
- The dataset is large, but the final processed CSV is not included in the GitHub repository.
- Reproducibility is limited because the notebook depends on a local file path and hidden preprocessing.
- At least two baseline models were trained and compared.
- Metrics should focus on Recall/F1/MCC/ROC-AUC, not only Accuracy, because false negatives are dangerous in phishing detection.
- If removing `malicious_probability` reduces performance significantly, the original model may rely on a potentially leakage-prone feature.